# dist-send-recv-pair — ex2: ring-pass via send/recv — every rank collects from all others

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dist-send-recv-pair`. Running the final beacon cell reports progress against the `Distributed: dist.send/recv pair` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: dist.send/recv pair` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dist-send-recv-pair`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dist-send-recv-pair"
DD_SUBTOPIC = "Distributed: dist.send/recv pair"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Ring-pass via send/recv — N-1 rotations around the ring

Ex1 used the linear `src loops sends to every other` pattern — fine for broadcast-from-one. The **ring-pass** is a different topology, where every rank pushes data to its right neighbor and receives from its left, repeated `world_size - 1` times. After all rotations, rank `r` has accumulated `world_size - 1` payloads from other ranks.

```python
left  = (rank - 1) % world_size
right = (rank + 1) % world_size
buf = my_payload.clone()
received = []
for _ in range(world_size - 1):
    dist.send(buf, dst=right)
    in_buf = t.zeros_like(buf)
    dist.recv(in_buf, src=left)
    received.append(in_buf.clone())
    buf = in_buf
```

**Why ring topology.** It's the structural backbone of `nccl`'s all-reduce — bandwidth scales O(1) per rank (each link carries the same volume regardless of `world_size`). Implementing it by hand once burns the pattern in.

**Deadlock trap.** `dist.send` is BLOCKING on gloo. If every rank calls `send` first and `recv` second, you must trust that the underlying transport buffers small messages — which gloo does for tensors below a few KB. For large tensors, use `dist.isend`/`irecv` and explicit `wait()`. This drill stays in the small-message regime.

### Exercise 2 — ring-pass via send/recv — every rank collects from all others

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply paired `dist.send(buf, dst=right)` + `dist.recv(in_buf, src=left)` in a loop of `world_size - 1` rotations to collect every other rank's payload at every rank — the ring-allreduce topology in its barest form.
> Keywords: send, recv, ring, rotation, topology
> ```

**KCs targeted:** `ring-neighbor-left-right-modular`, `n-minus-1-rotations`

Implement `ex2_ring_collect(rank, world_size, dist_module, my_payload)`. The ring-pass collector:

1. Compute neighbors: `left = (rank - 1) % world_size`, `right = (rank + 1) % world_size`.
2. Initialize `buf = my_payload.clone()` (the payload moving around the ring) and `received = []` (the list of payloads this rank has seen).
3. Loop `world_size - 1` times:
   a. `dist_module.send(buf, dst=right)` — push to right neighbor.
   b. Allocate fresh `in_buf = t.zeros_like(buf)`.
   c. `dist_module.recv(in_buf, src=left)` — receive from left.
   d. Append `in_buf.clone()` to `received`.
   e. Set `buf = in_buf` so the next rotation passes the just-received payload onward.
4. Return `received` — a list of `world_size - 1` tensors, in the order they were received.

After `world_size - 1` rotations, every rank has seen every OTHER rank's original payload (in some order; the order depends on rank, which we don't assert).

Input: `rank`, `world_size` — ints; `dist_module` — torch.distributed or mock; `my_payload` — a 1-D float tensor unique to this rank.
Output: `list[Tensor]` of length `world_size - 1`.

In [ ]:
def ex2_ring_collect(rank: int, world_size: int, dist_module, my_payload: Tensor) -> list:
    left = (rank - 1) % world_size
    right = (rank + 1) % world_size
    buf = my_payload.clone()
    received = []
    for _ in range(world_size - 1):
        dist_module.send(buf, dst=right)
        in_buf = t.zeros_like(buf)
        dist_module.recv(in_buf, src=left)
        received.append(in_buf.clone())
        buf = in_buf
    return received


<details><summary>Solution</summary>

```python
def ex2_ring_collect(rank: int, world_size: int, dist_module, my_payload: Tensor) -> list:
    left = (rank - 1) % world_size
    right = (rank + 1) % world_size
    buf = my_payload.clone()
    received = []
    for _ in range(world_size - 1):
        dist_module.send(buf, dst=right)
        in_buf = t.zeros_like(buf)
        dist_module.recv(in_buf, src=left)
        received.append(in_buf.clone())
        buf = in_buf
    return received
```

**Why `world_size - 1` rotations.** After 1 rotation, every rank has its left-neighbor's original payload. After 2 rotations, every rank has its left-left-neighbor's payload (the prior payload was passed on). After `W-1` rotations, every OTHER rank's payload has visited every rank exactly once.

**`buf = in_buf` instead of `buf.copy_(in_buf)`.** Either works semantically. The rebind is more idiomatic in Python and lets the garbage collector reclaim the old buffer; the in-place copy is marginally faster but allocates an extra tensor at `zeros_like`.

**Deadlock-safety on real gloo.** Every rank calls send THEN recv in the same order, but gloo's send is non-blocking for small tensors — it copies to an internal buffer and returns. The recv then drains. For large tensors (>~64KB), gloo blocks the send until the matching recv posts — which would deadlock this code. The ring needs `isend`/`irecv` for production scale.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()